In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import os
from openai import OpenAI
from datasets import load_dataset, DatasetDict
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorWithPadding
from pgkd import PGKD
import wandb


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# ➊ Train baseline supervised model_0 on the labelled subset

In [ ]:
model_0 = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=cfg.num_labels,
    id2label=id2label,
    label2id=label2id,
)

baseline_args = TrainingArguments(
    output_dir="checkpoints/model_0",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer0 = Trainer(
    model=model_0,
    args=baseline_args,
    train_dataset=train,
    eval_dataset=valid,
    compute_metrics=compute_metrics_factory(cfg.num_labels),
)
trainer0.train()
print("🏁  Baseline (model_0) validation:", trainer0.evaluate())

# ➋ Run PGKD starting from the fine‑tuned baseline

In [ ]:
pgkd = PGKD(
    tokenizer=tokenizer,
    model=model_0,            # start from supervised baseline
    num_labels=cfg.num_labels,
    initial_dataset=train,
    val_dataset=valid,
    dataset_class_taxonomy=id2label,
    teacher=teacher,
    id2label=id2label,
    label2id=label2id,
)

pgkd.train()